# ARC-AGI Solver — Phase A: RE-ARC Training with Augmentation

Trains the decoder-only transformer on RE-ARC synthetic data (1,000 examples
per task) with colour-permutation augmentation applied during training.

**Stratified split** (from `data/task_split.json`, category-aware):
- **Train**: 278 tasks — RE-ARC data used for training
- **Val**: 81 tasks — ARC leave-one-out exact match, used for checkpoint selection
- **Eval**: 41 tasks — held out completely, never seen during training or selection

This is the **Phase A** improvement over the `all_400_arc` baseline (original ARC
pairs only, ~3–7 per task, no held-out validation).  Both models share the same
architecture and evaluation pipeline so results are directly comparable.

Checkpoint name: `transformer_crearc_aug_278_best.pt`

**Cell order:** 1 → 2 → 3 → 4 → 5 → 6 → 7

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training on CPU will be extremely slow.')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE          = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR      = f'{DRIVE_BASE}/checkpoints'
DRIVE_TOKENIZED_DIR = f'{DRIVE_BASE}/tokenized'

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_TOKENIZED_DIR).mkdir(parents=True, exist_ok=True)

print(f'Checkpoint dir:  {DRIVE_CKPT_DIR}')
print(f'Tokenized dir:   {DRIVE_TOKENIZED_DIR}')

In [ ]:
# ── Cell 3: Clone repo and download ARC training data ───────────────────────
import os, io, subprocess, urllib.request, zipfile
from pathlib import Path

REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
REPO_DIR    = f'/content/{REPO}'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git', REPO_DIR],
                   check=True)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
log = subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-3'],
                     capture_output=True, text=True)
print(log.stdout)

# Download ARC training data
DATA_DIR  = Path('data')
train_dir = DATA_DIR / 'training'
if not train_dir.exists() or len(list(train_dir.glob('*.json'))) < 400:
    print('Downloading ARC training data...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        dest = DATA_DIR / 'training'
        dest.mkdir(exist_ok=True)
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list(train_dir.glob('*.json')))} tasks")
else:
    print(f"ARC training data already present ({len(list(train_dir.glob('*.json')))} tasks)")

In [ ]:
# ── Cell 4: Download RE-ARC synthetic data ──────────────────────────────────
# 400 tasks × 1,000 examples each → data/re_arc/  (~110 MB zip)
import subprocess, sys
from pathlib import Path

RE_ARC_DIR = Path('data/re_arc')
if RE_ARC_DIR.exists() and len(list(RE_ARC_DIR.glob('*.json'))) >= 400:
    print(f'RE-ARC already present ({len(list(RE_ARC_DIR.glob("*.json")))} tasks)')
else:
    print('Downloading RE-ARC synthetic data...')
    result = subprocess.run([sys.executable, 'scripts/download_re_arc.py'],
                            capture_output=True, text=True)
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    print(f'RE-ARC: {len(list(RE_ARC_DIR.glob("*.json")))} tasks')

In [ ]:
# ── Cell 5: Configuration ────────────────────────────────────────────────────
# Loads the category-stratified 70/20/10 task split from data/task_split.json.
#
# Train  (278 tasks): RE-ARC synthetic data (1,000 examples/task)
# Val    (81 tasks): ARC leave-one-out exact match — checkpoint criterion
# Eval   (41 tasks): held out completely; never used for training or selection
#
# Compare against all_400_arc (original ARC pairs, no stratification).
import json
from pathlib import Path

RUN_NAME        = 'rearc_aug_278'  # checkpoint: transformer_crearc_aug_278_best.pt
K_CONTEXT       = 3
D_MODEL         = 512
N_HEADS         = 8
N_LAYERS        = 6
LR              = 3e-4
WARMUP_EPOCHS   = 5
EPOCHS          = 300
STEPS_PER_EPOCH = 400
MAX_TOKENS      = 96000
SAVE_EVERY      = 25
LOG_EVERY       = 1
LOG_FILE        = f'results/train_{RUN_NAME}.log'

split        = json.loads(Path('data/task_split.json').read_text())
TASK_IDS     = split['train']   # 278 tasks — RE-ARC training
VAL_TASK_IDS = split['val']     # 81 tasks — checkpoint selection

print(f'Run:         {RUN_NAME}')
print(f'Train tasks: {len(TASK_IDS)}  (RE-ARC data, 1000 examples each)')
print(f'Val tasks:   {len(VAL_TASK_IDS)}  (ARC LOO exact match — checkpoint criterion)')
print(f'Eval tasks:  {len(split["eval"])}  (held out completely)')
print(f'Epochs:      {EPOCHS} × {STEPS_PER_EPOCH} steps = {EPOCHS*STEPS_PER_EPOCH:,} total')
print(f'd_model:     {D_MODEL},  heads: {N_HEADS},  layers: {N_LAYERS},  k_ctx: {K_CONTEXT}')
print(f'LR:          {LR},  warmup: {WARMUP_EPOCHS} epochs')

In [ ]:
# ── Cell 6: Pre-tokenise RE-ARC examples ────────────────────────────────────
# Encodes every RE-ARC pair once → data/tokenized/<task_id>.npz
# Synced to/from Drive so it only runs once across sessions.
# Only tokenises tasks in TASK_IDS (the training split).
import subprocess, sys, shutil
from pathlib import Path

local_tok = Path('data/tokenized')
drive_tok = Path(DRIVE_TOKENIZED_DIR)
local_tok.mkdir(parents=True, exist_ok=True)

# Restore any files already on Drive
restored = 0
for p in drive_tok.glob('*.npz'):
    dest = local_tok / p.name
    if not dest.exists():
        shutil.copy2(p, dest)
        restored += 1
if restored:
    print(f'Restored {restored} .npz file(s) from Drive.')

# Tokenise anything still missing from the training split
already = {p.stem for p in local_tok.glob('*.npz')}
missing = [tid for tid in TASK_IDS if tid not in already]

if not missing:
    print(f'All {len(TASK_IDS)} training tasks already tokenised.')
else:
    print(f'Tokenising {len(missing)} tasks...')
    result = subprocess.run(
        [sys.executable, 'scripts/pretokenize.py', '--tasks'] + missing,
        capture_output=True, text=True,
    )
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

# Sync new files to Drive
synced = 0
for p in local_tok.glob('*.npz'):
    dest = drive_tok / p.name
    if not dest.exists():
        shutil.copy2(p, dest)
        synced += 1
if synced:
    print(f'Saved {synced} new .npz file(s) to Drive.')
print(f'Tokenized files on disk: {len(list(local_tok.glob("*.npz")))}')

In [ ]:
# ── Cell 7: Run training ─────────────────────────────────────────────────────
# Data source: RE-ARC (1,000 synthetic examples per training task).
# Augmentation: colour permutation applied per batch (via --data-source rearc).
# Checkpoint criterion: highest ARC LOO exact match on VAL_TASK_IDS.
# Val tasks are NOT in the training pool — clean, uncontaminated validation.
import subprocess, sys

cmd = [
    sys.executable, '-u', 'scripts/train_transformer.py',
    '--data-source',      'rearc',          # RE-ARC synthetic data
    '--task-ids',         *TASK_IDS,
    '--val-arc-task-ids', *VAL_TASK_IDS,
    '--run-name',         RUN_NAME,
    '--epochs',           str(EPOCHS),
    '--steps-per-epoch',  str(STEPS_PER_EPOCH),
    '--k-context',        str(K_CONTEXT),
    '--d-model',          str(D_MODEL),
    '--n-heads',          str(N_HEADS),
    '--n-layers',         str(N_LAYERS),
    '--lr',               str(LR),
    '--warmup-epochs',    str(WARMUP_EPOCHS),
    '--max-tokens',       str(MAX_TOKENS),
    '--max-seq-len',      '6000',
    '--save-every',       str(SAVE_EVERY),
    '--log-every',        str(LOG_EVERY),
    '--log',              LOG_FILE,
    '--ckpt-dir',         DRIVE_CKPT_DIR,
]

print(f'Checkpoints → {DRIVE_CKPT_DIR}')
print(f'Log         → {LOG_FILE}')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 8: View training log (last 40 lines) ───────────────────────────────
from pathlib import Path
log = Path(LOG_FILE)
if log.exists():
    lines = log.read_text().splitlines()
    print(f'Log: {log}  ({len(lines)} lines)')
    print('\n'.join(lines[-40:]))
else:
    print('Log file not found — training may not have started yet.')